In [6]:
import torch
import torch.nn as nn
import joblib
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Load Models
logreg = joblib.load('models/logistic_regression_model.pkl')
knn = joblib.load('models/knn_model.pkl')
svm = joblib.load('models/svm_model.pkl')


In [ ]:
class ABClassifierCNN(nn.Module):
    def __init__(self):
        super(ABClassifierCNN, self).__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )
        self.fc1 = nn.Linear(64*7*7, 128)
        self.fc2 = nn.Linear(128, 2)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

cnn_model = ABClassifierCNN().to(device)
cnn_model.load_state_dict(torch.load('models/ab_classifier_cnn.pt', map_location=device))
cnn_model.eval()


In [ ]:
from PIL import Image, ImageOps
import torch
import torch.nn as nn
from torchvision import transforms
import matplotlib.pyplot as plt
import numpy as np
import joblib

image_path = 'images/balon.jpg'  

# Load image in grayscale
img = Image.open(image_path).convert('L')

# Invert the image (white ↔ black)
img = ImageOps.invert(img)

# Rotate the image 90 degrees
img = img.transpose(Image.ROTATE_90)

# Show preprocessed image
plt.imshow(img, cmap='gray')
plt.title("Preprocessed Image")
plt.axis('off')
plt.show()

# Resize and prepare image for ML models
img_ml = img.resize((28, 28))
img_ml = np.array(img_ml).reshape(1, -1)  
img_ml = img_ml.astype('float32') / 255.0  

# Transform for CNN input
transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  
])

img_cnn = transform(img).unsqueeze(0) 

# CNN Model Definition
class ABClassifierCNN(nn.Module):
    def __init__(self):
        super(ABClassifierCNN, self).__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 2)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load CNN model
cnn_model = ABClassifierCNN().to(device)
cnn_model.load_state_dict(torch.load('models/ab_classifier_cnn.pt', map_location=device))
cnn_model.eval()

# Predictions from traditional models
pred_logreg = logreg.predict(img_ml)[0]
pred_knn = knn.predict(img_ml)[0]
pred_svm = svm.predict(img_ml)[0]

# Prediction from CNN model
with torch.no_grad():
    output = cnn_model(img_cnn.to(device))
    _, pred_cnn = torch.max(output, dim=1)
    pred_cnn = pred_cnn.item()

# Label map for output
label_map = {0: 'A', 1: 'B'}

# Print predictions
print("\n Predictions")
print(f"Logistic Regression Prediction: {label_map.get(pred_logreg, 'Unknown')}")
print(f"KNN Prediction: {label_map.get(pred_knn, 'Unknown')}")
print(f"SVM Prediction: {label_map.get(pred_svm, 'Unknown')}")
print(f"CNN Prediction: {label_map.get(pred_cnn, 'Unknown')}")
